[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Dpkt](Dpkt.md) | [Pyarrow](Pyarrow.md) | [Tasks](Tasks.md) | [Task 7 Guide](Task-hilbert-map.md) | flow compare | [flow analysis](flow_analysis.ipynb) | [Slides](slides/)

### Network Infrastructure Data Science (NIDS) Module
---
# Step 1. Creating Flow records from Network Telescope Pcaps

Make sure your instance has at least 16GB memory or above (recommend 24GB).

The traffic data we provided is a small sample of traffic data collected by one of the /16 in UCSD Network Telescope. The destination IPs are anonymized into 10.0.0.0/8. The source IPs are the original value.

`PCAP_A` and `PCAP_B` are two short samples collected on June 26, 2026 at 08:53 and 09:00 UTC, respectively (about 7 minutes apart); the traffic surge appears in `PCAP_B`.

Complete Task 1 and Task 2 to create two Apache Parquet files containing flow records summarizing the traffic for investigating a scanning event.

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

## Imports

In [ ]:
%pip install -q dpkt pandas pyarrow pybgpkit-parser pelicanfs

In [ ]:
import contextlib
import gc
import gzip
import io
import pickle
import socket
import urllib.parse
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
import struct

import dpkt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import pybgpkit_parser as bgpkit
from pelicanfs.core import OSDFFileSystem

In [ ]:
# NRP Ceph s3 endpoints pcap files
PCAP_A = "http://rook-ceph-rgw-nautiluss3.rook/caida/ucsd-nt/sample_062026/ucsd-nt-sub.1782463980.anon.pcap.gz"  # local path or http(s) URL
PCAP_B = "http://rook-ceph-rgw-nautiluss3.rook/caida/ucsd-nt/sample_062026/ucsd-nt-sub.1782464400.anon.pcap.gz"  # local path or http(s) URL
TOP_N = 10

# Names for the parquet files to be written out. flow_analysis.ipynb reads these
# same two names back in, so if you change them here, change them there too.
# They must also be different from each other.
FLOWS_A_OUT = "ucsd-nt-sub.1782463980.flow.parquet"
FLOWS_B_OUT = "ucsd-nt-sub.1782464400.flow.parquet"

## Data model (provided)

In [ ]:
@dataclass
class FlowStats:
    proto: str
    total_packets: int = 0
    total_bytes: int = 0
    first_seen: float = None
    last_seen: float = None
    tcp_flags_seen: set = field(default_factory=set)

## Streaming reader (provided)

In [ ]:
def _open_binary_stream(source):
    """Open source for reading raw bytes.

    source may be a local file path or an http(s) URL (e.g. a presigned
    S3 link). Either way this returns an unbuffered-content stream
    supporting .read(); it is not decompressed yet.
    """
    parsed = urllib.parse.urlparse(source)
    if parsed.scheme in ("http", "https"):
        return urllib.request.urlopen(source)
    return open(source, "rb")


def iter_packets(source):
    """Yield (timestamp, raw_frame_bytes) tuples, one per packet.

    source may be a local path or an http(s) URL. The captures for this
    assignment are gzip-compressed, so the stream is sniffed for the
    gzip magic bytes (\\x1f\\x8b) and transparently decompressed - this
    is more reliable than checking the filename/URL suffix, since a
    presigned S3 URL typically has a query string after ".gz" (e.g.
    "...capture_a.pcap.gz?X-Amz-Signature=...").

    dpkt.pcap.Reader parses only the pcap global/record headers as it
    goes; the raw frame bytes are handed back unparsed so no per-packet
    object graph is built until flow_key() decides it needs one, and
    gzip.GzipFile decompresses one chunk at a time rather than
    inflating the whole file up front. This keeps memory flat
    regardless of file size, whether reading from disk or over HTTP.
    """
    with contextlib.closing(_open_binary_stream(source)) as raw:
        buffered = raw if hasattr(raw, "peek") else io.BufferedReader(raw)
        is_gzip = buffered.peek(2)[:2] == b"\x1f\x8b"
        stream = gzip.GzipFile(fileobj=buffered) if is_gzip else buffered
        reader = dpkt.pcap.Reader(stream)
        for ts, buf in reader:
            yield ts, buf

## Parsing packets with dpkt

Task 1 turns one raw frame into a flow key, so it helps to know the shape of what `dpkt` hands you. A parsed packet is a chain of layers, each reachable through the previous layer's `.data`:

```
Ethernet ─.data→ IP ─.data→ TCP / UDP / ICMP
```

You decode the outermost layer and walk inward, checking the type at each step (a telescope sees non-IPv4 frames too, so never assume):

```python
eth = dpkt.ethernet.Ethernet(buf)   # decode the frame
ip  = eth.data                      # could be IP, ARP, IPv6, ...
if isinstance(ip, dpkt.ip.IP):      # IPv4?
    transport = ip.data             # could be TCP, UDP, ICMP, ...
```

- `ip.src` / `ip.dst` are raw 4-byte addresses — convert with `socket.inet_ntoa(...)`.
- TCP and UDP expose `.sport` / `.dport`; ICMP has no ports (use `.type` / `.code`).
- TCP `.flags` is a bitfield — the provided `_tcp_flags_str()` decodes it for you.

**See the [dpkt guide](Dpkt.md)** for the full layered model, address conversion, the TCP-flag bitfield, why dpkt (not scapy) at telescope scale, and the parsing best practices — including guarding against malformed frames, which the `flow_key` skeleton below already does for you.

## Task 1: Flow Key Extraction

In [ ]:
_TCP_FLAG_BITS = (
    (dpkt.tcp.TH_SYN, "S"),
    (dpkt.tcp.TH_ACK, "A"),
    (dpkt.tcp.TH_FIN, "F"),
    (dpkt.tcp.TH_RST, "R"),
    (dpkt.tcp.TH_PUSH, "P"),
    (dpkt.tcp.TH_URG, "U"),
)


def _tcp_flags_str(flags):
    return "".join(letter for bit, letter in _TCP_FLAG_BITS if flags & bit) or "."


def flow_key(buf):
    """Parse one raw Ethernet frame and compute its 5-tuple flow key.

    Returns (key, tcp_flags) or None if the frame has no IPv4 layer or
    cannot be parsed, where key = (ip_src, port_src, ip_dst, port_dst,
    proto) and tcp_flags is a short string like "S"/"SA"/"FA" for TCP
    packets, or None for non-TCP traffic.

    Note: this assumes an Ethernet link layer (dpkt.pcap.Reader does
    not auto-select a decoder based on the pcap linktype), and treats
    IPv6 as out of scope - IPv6 frames fall through to the "not IP"
    branch below.
    """
    try:
        # student_code_start --------------------------------------------
        # YOUR CODE HERE  (see the dpkt guide - Dpkt.md - for the walkthrough)
        #
        # Hints:
        #   - dpkt.ethernet.Ethernet(buf) parses the Ethernet frame; its
        #     .data is the next layer up. Check isinstance(ip, dpkt.ip.IP)
        #     to make sure it's IPv4 - return None if not.
        #   - socket.inet_ntoa(ip.src) / socket.inet_ntoa(ip.dst) convert
        #     the raw 4-byte addresses to dotted-decimal strings.
        #   - Check isinstance(ip.data, dpkt.tcp.TCP) / dpkt.udp.UDP /
        #     dpkt.icmp.ICMP to find the transport layer:
        #       * TCP/UDP have .sport / .dport
        #       * ICMP has no ports - use .type / .code in their place
        #       * anything else: proto = "OTHER", sport = dport = 0
        #   - For TCP, build tcp_flags with the provided _tcp_flags_str().
        #   - Return ((ip_src, sport, ip_dst, dport, proto), tcp_flags).
        #
        # The try/except around this block is a best practice (see Dpkt.md
        # section 7): a truncated or malformed frame makes dpkt raise, and
        # we skip that one packet instead of aborting the whole capture.
        raise NotImplementedError
        # student_code_end ----------------------------------------------
    except (dpkt.dpkt.NeedData, dpkt.dpkt.UnpackError):
        return None

## Flow map construction (provided)

In [ ]:
def build_flow_map(source):
    flow_map = {}
    total_packets = 0
    start_time = None
    end_time = None

    for ts, buf in iter_packets(source):
        total_packets += 1
        result = flow_key(buf)
        if result is None:
            continue
        key, tcp_flags = result

        pkt_len = len(buf)
        ts = float(ts)

        if start_time is None or ts < start_time:
            start_time = ts
        if end_time is None or ts > end_time:
            end_time = ts

        stats = flow_map.get(key)
        if stats is None:
            stats = FlowStats(proto=key[4])
            flow_map[key] = stats

        stats.total_packets += 1
        stats.total_bytes += pkt_len
        if stats.first_seen is None or ts < stats.first_seen:
            stats.first_seen = ts
        if stats.last_seen is None or ts > stats.last_seen:
            stats.last_seen = ts

        if tcp_flags is not None:
            stats.tcp_flags_seen.add(tcp_flags)


    meta = {
        "total_packets": total_packets,
        "start_time": start_time,
        "end_time": end_time,
    }
    return flow_map, meta

## Task 2: Per-Capture Summary

In [ ]:
def summarize_flow_map(flow_map, meta, top_n=10, label=""):
    # student_code_start ------------------------------------------------
    # YOUR CODE HERE
    # Compute:
    #   total_flows   - number of distinct flows in flow_map
    #   total_packets - sum of total_packets across all flows
    #   total_bytes   - sum of total_bytes across all flows
    #   proto_counts  - dict mapping proto -> number of flows with that proto
    total_flows = None
    total_packets = None
    total_bytes = None
    proto_counts = None
    # student_code_end --------------------------------------------------

    # the following printout is provided
    print(f"\n=== Summary: {label} ===")
    print(f"packets read:       {meta['total_packets']}")
    print(f"total flows:        {total_flows}")
    print(f"total flow packets: {total_packets}")
    print(f"total flow bytes:   {total_bytes}")

    print("protocol breakdown:")
    for proto, count in sorted(proto_counts.items(), key=lambda kv: -kv[1]):
        print(f"  {proto:6s}: {count}")

    top_flows = sorted(flow_map.items(), key=lambda kv: -kv[1].total_bytes)[:top_n]
    print(f"top {top_n} flows by bytes:")
    for key, stats in top_flows:
        ip_src, port_src, ip_dst, port_dst, proto = key
        print(f"  {ip_src}:{port_src} -> {ip_dst}:{port_dst} [{proto}] "
              f"packets={stats.total_packets} bytes={stats.total_bytes}")

### Saving flow records to parquet

Writes straight from `flow_map` via `pyarrow` - no intermediate pandas `DataFrame` is built while `flow_map` is still alive. See the [pyarrow guide](Pyarrow.md) for why the write path is built this way, and how the analysis notebook reads it back.

In [ ]:
def write_flow_map_parquet(flow_map, path):
    """Persist a flow map directly to a parquet file via pyarrow.

    This builds columnar arrays straight from flow_map and writes them
    with pyarrow - it does not go through an in-memory pandas
    DataFrame. flow_map (built while streaming the raw pcap) and any
    DataFrame view of the same data are two separate copies of the
    same information; going flow_map -> parquet -> DataFrame (via
    pd.read_parquet, see below) means only one copy is ever resident
    at a time, and the flow_map can be freed as soon as this returns.

    Columns: ip_src, port_src, ip_dst, port_dst, proto, total_packets,
    total_bytes, first_seen, last_seen, tcp_flags_seen (joined into a
    single comma-separated string, since a Python set has no parquet
    representation).
    """
    columns = {
        "ip_src": [], "port_src": [], "ip_dst": [], "port_dst": [], "proto": [],
        "total_packets": [], "total_bytes": [], "first_seen": [],
        "last_seen": [], "tcp_flags_seen": [],
    }
    for key, stats in flow_map.items():
        columns["ip_src"].append(key[0])
        columns["port_src"].append(key[1])
        columns["ip_dst"].append(key[2])
        columns["port_dst"].append(key[3])
        columns["proto"].append(key[4])
        columns["total_packets"].append(stats.total_packets)
        columns["total_bytes"].append(stats.total_bytes)
        columns["first_seen"].append(stats.first_seen)
        columns["last_seen"].append(stats.last_seen)
        columns["tcp_flags_seen"].append(",".join(sorted(stats.tcp_flags_seen)))

    pq.write_table(pa.table(columns), path)

## Run it

Process each capture **one at a time**: build its flow map, summarize it, persist it to parquet, then free it before moving to the next capture. Neither capture's raw flow_map is ever resident in memory at the same time as the other's.

In [ ]:
def process_capture(source, out_path, top_n):
    """Build one capture's flow map, report on it, persist it to
    parquet, then drop the flow map from memory.

    After this returns, nothing about this capture's individual
    packets or flows is still resident - only the parquet file on
    disk. This is what lets the run cell below hold at most one
    capture's raw flow_map in memory at a time, no matter how large
    the two pcaps are.
    """
    flow_map, meta = build_flow_map(source)
    summarize_flow_map(flow_map, meta, top_n=top_n, label=source)
    write_flow_map_parquet(flow_map, out_path)
    del flow_map, meta
    gc.collect()

In [ ]:
process_capture(PCAP_A, FLOWS_A_OUT, TOP_N)

In [ ]:
process_capture(PCAP_B, FLOWS_B_OUT, TOP_N)

### Question 1

Which transport protocol accounts for the most **flows** in both PCAPs? (The protocol breakdown printed above counts flows, not packets - the two can point different ways.)

YOUR ANSWER HERE

### Question 2

Among the top-N flows by bytes in both PCAPs, which TCP destination port was targeted the most? What application is conventionally assigned to that port, and why is it a popular target?

YOUR ANSWER HERE

### Question 3

Which PCAP contains more traffic in terms of number of packets, and by how much? Is there a significant increase in traffic contributed by the top-N flows by bytes?

YOUR ANSWER HERE